<a href="https://colab.research.google.com/github/flathfk/Bootcamp_Study/blob/main/260327_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JOIN 연산의 이해와 활용

## 1. 학습 개요

- 관계 대수(Relational Algebra)의 결합 연산 원리를 이해한다.
- MariaDB 환경에서 다양한 JOIN 유형의 문법과 동작 차이를 습득한다.
- 데이터 무결성과 정규화 관점에서 JOIN의 필요성을 고찰한다.

### 1.1 조인(JOIN)의 본질적 정의

조인이란 서로 독립적으로 존재하는 **두 개 이상의 릴레이션**(Table)을 **공통 속성**(Join Key)을 매개로 하여 수평적으로 **결합**(Horizontal Combination)함으로써, 논리적으로 하나의 거대한 정보 집합으로 확장하는 연산입니다.

### 1.2 조인의 핵심 구성 요소

- **1.2.1 공통 키 (Common Key):** 두 테이블을 연결하는 ' **이정표** '입니다. 주로 부모 테이블의 기본 키(PK)와 자식 테이블의 외래 키(FK)가 이 역할을 수행합니다.
- **1.2.2 튜플 간 결합 (Tuple Matching):** 조인 조건(`ON`)을 만족하는 각 테이블의 행(Row)들이 서로 ' **짝을 찾는 과정** '입니다.
- **1.2.3 정보의 확장 (Data Extension):** 정규화로 인해 수직으로 분리된 속성들을 다시 옆으로 길게 붙여 ' **의미 있는 하나의 정보셋** '으로 복원하는 행위입니다.

### 1.3 조인 연산의 논리적 구분 동작

- **Step 1 (연결):** 공통 키를 통해 두 테이블 사이의 연관 관계를 식별합니다.
- **Step 2 (합성):** 일치하는 튜플끼리 결합하여 새로운 가상 릴레이션을 생성합니다.
- **Step 3 (추출):** 합성된 데이터 중 사용자가 원하는 속성(Column)만 선택하여 출력합니다.

> JOIN은 쪼개진 퍼즐 조각을 공통된 그림(Key)을 보고 다시 맞춰서 커다란 그림(확장)을 만드는 과정
>

## 2. JOIN의 이론적 배경

### 2.1 관계 대수와 카테시안 곱 (Cartesian Product)

모든 JOIN의 기초는 두 릴레이션(Relation) R과 S의 모든 가능한 순서쌍을 만드는 **카테시안 곱**( $R \times S$ )에서 시작합니다. JOIN은 이 방대한 결과 집합에 **선택 조건**(Selection Condition)을 결합하여 유의미한 부분 집합을 추출하는 연산입니다.

### 2.2 연결 속성 (Join Attribute)

두 테이블을 연결하는 공통 속성입니다. 일반적으로 한 테이블의 **기본 키**(Primary Key)와 다른 테이블의 **외래 키(Foreign Key)** 간의 관계를 통해 데이터의 일관성을 유지하며 결합합니다.

## 3. JOIN 종류

### 3.1 Inner Join

두 릴레이션에서 조인 조건을 만족하는(매칭되는) 튜플만 결과에 포함시킵니다.

- **세타 조인(Theta Join):** 임의의 비교 연산자( $=, <, >, \dots$  )를 사용하는 조인.
- **동등 조인(Equi-Join):** 비교 연산자가 '='인 조인.
- **자연 조인(Natural Join):** 두 테이블에서 이름이 같은 모든 컬럼을 대상으로 동등 조인을 수행하며, 중복 컬럼을 제거합니다.

### 3.2 Outer Join

조인 조건을 만족하지 않는 튜플도 결과에 포함시키기 위해 사용합니다. 매칭되지 않는 부분은 `NULL` 값으로 채워집니다.

- **Left Outer Join:** 왼쪽 릴레이션 $R$의 모든 튜플을 보존합니다.
- **Right Outer Join:** 오른쪽 릴레이션 $S$의 모든 튜플을 보존합니다.
- **Full Outer Join:** 양쪽 릴레이션의 모든 튜플을 보존합니다. (MariaDB에서는 `UNION`을 통해 구현)

## 4. SQL 구문 상세

### 4.1 기본 구문 구조

```sql
SELECT <컬럼 리스트>
FROM 테이블1 [AS 별칭]
[INNER | LEFT | RIGHT] JOIN 테이블2 [AS 별칭]
    ON 테이블1.공통컬럼 = 테이블2.공통컬럼
[WHERE 필터조건];
```

### 4.2 주요 구문 변형

1. **USING 절:** 조인 대상 컬럼명이 양쪽 테이블에서 동일할 때 사용합니다.
    - `JOIN table2 USING (column_name)`
2. **Self Join:** 순환적 관계(Recursive Relationship)를 표현할 때 하나의 테이블을 서로 다른 별칭으로 조인합니다.
    - 예: 조직도(사원-관리자)

In [1]:
%%bash
apt-get install -y mariadb-server > /dev/null 2>&1
service mariadb start

 * Starting MariaDB database server mariadbd
   ...done.


In [3]:
%%bash

mysql -u root << 'EOF'
CREATE DATABASE IF NOT EXISTS bankdb;
CREATE USER IF NOT EXISTS 'sorah'@'localhost' IDENTIFIED BY '1234';
GRANT ALL PRIVILEGES ON bankdb.* TO 'sorah'@'localhost';
FLUSH PRIVILEGES;

USE bankdb;

CREATE TABLE IF NOT EXISTS customers (
    cust_id   INT PRIMARY KEY,
    cust_name VARCHAR(50) NOT NULL,
    grade     VARCHAR(10)
);

CREATE TABLE IF NOT EXISTS accounts (
    acc_no   VARCHAR(20) PRIMARY KEY,
    cust_id  INT,
    acc_type VARCHAR(20),
    balance  DECIMAL(15, 2),
    FOREIGN KEY (cust_id) REFERENCES customers(cust_id)
);

CREATE TABLE IF NOT EXISTS transactions (
    tx_id   INT AUTO_INCREMENT PRIMARY KEY,
    acc_no  VARCHAR(20),
    tx_type VARCHAR(10),
    amount  DECIMAL(15, 2),
    tx_date DATETIME,
    FOREIGN KEY (acc_no) REFERENCES accounts(acc_no)
);

INSERT IGNORE INTO customers VALUES
(1, '박지성', 'VIP'),
(2, '손흥민', 'GOLD'),
(3, '김연아', 'SILVER'),
(4, '이강인', 'SILVER');

INSERT IGNORE INTO accounts VALUES
('110-001', 1, 'SAVINGS', 5000000),
('110-002', 1, 'CHECKING', 120000),
('110-003', 2, 'SAVINGS', 3000000),
('110-004', 3, 'CHECKING', 50000);

INSERT IGNORE INTO transactions (acc_no, tx_type, amount, tx_date) VALUES
('110-001', 'DEPOSIT', 1000000, '2026-03-01 10:00:00'),
('110-001', 'WITHDRAW', 500000, '2026-03-02 14:20:00'),
('110-003', 'DEPOSIT', 200000, '2026-03-03 09:15:00');
EOF

In [4]:
%%bash

mysql -u root --table bankdb << 'EOF'
SELECT * FROM customers;
SELECT * FROM accounts;
SELECT * FROM transactions;
EOF

+---------+-----------+--------+
| cust_id | cust_name | grade  |
+---------+-----------+--------+
|       1 | 박지성    | VIP    |
|       2 | 손흥민    | GOLD   |
|       3 | 김연아    | SILVER |
|       4 | 이강인    | SILVER |
+---------+-----------+--------+
+---------+---------+----------+------------+
| acc_no  | cust_id | acc_type | balance    |
+---------+---------+----------+------------+
| 110-001 |       1 | SAVINGS  | 5000000.00 |
| 110-002 |       1 | CHECKING |  120000.00 |
| 110-003 |       2 | SAVINGS  | 3000000.00 |
| 110-004 |       3 | CHECKING |   50000.00 |
+---------+---------+----------+------------+
+-------+---------+----------+------------+---------------------+
| tx_id | acc_no  | tx_type  | amount     | tx_date             |
+-------+---------+----------+------------+---------------------+
|     1 | 110-001 | DEPOSIT  | 1000000.00 | 2026-03-01 10:00:00 |
|     2 | 110-001 | WITHDRAW |  500000.00 | 2026-03-02 14:20:00 |
|     3 | 110-003 | DEPOSIT  |  200000.00 | 2026-0

### 실습 1. 보유 계좌가 있는 고객 명단 추출 (INNER JOIN)

**목적:** 고객 정보와 계좌 정보를 결합하여, 실제 계좌를 개설한 이력이 있는 고객의 이름과 계좌 번호를 조회합니다.

In [5]:
%%bash

mysql -u root --table bankdb << 'EOF'
SELECT c.cust_name, a.acc_no, a.acc_type
FROM customers c
INNER JOIN accounts a ON c.cust_id = a.cust_id;
EOF

+-----------+---------+----------+
| cust_name | acc_no  | acc_type |
+-----------+---------+----------+
| 박지성    | 110-001 | SAVINGS  |
| 박지성    | 110-002 | CHECKING |
| 손흥민    | 110-003 | SAVINGS  |
| 김연아    | 110-004 | CHECKING |
+-----------+---------+----------+


### 실습 2. 전체 고객의 계좌 보유 현황 조회 (LEFT OUTER JOIN)

**목적:** 계좌가 없는 신규 고객을 포함하여 모든 고객의 리스트를 출력하고, 계좌가 없는 경우 `NULL`로 표시합니다.

In [6]:
%%bash

mysql -u root --table bankdb << 'EOF'
SELECT c.cust_name, c.grade, a.acc_no
FROM customers c
LEFT JOIN accounts a ON c.cust_id = a.cust_id;
EOF

+-----------+--------+---------+
| cust_name | grade  | acc_no  |
+-----------+--------+---------+
| 박지성    | VIP    | 110-001 |
| 박지성    | VIP    | 110-002 |
| 손흥민    | GOLD   | 110-003 |
| 김연아    | SILVER | 110-004 |
| 이강인    | SILVER | NULL    |
+-----------+--------+---------+


### 실습 3. 특정 고객(VIP)의 거래 내역 상세 조회 (3-Way JOIN)

**목적:** 고객명, 계좌번호, 거래유형, 거래금액을 한 번에 조회합니다. 3개의 테이블을 체인 형태로 연결합니다.

In [7]:
%%bash

mysql -u root --table bankdb << 'EOF'
SELECT c.cust_name, a.acc_no, t.tx_type, t.amount
FROM customers c
JOIN accounts a ON c.cust_id = a.cust_id
JOIN transactions t ON a.acc_no = t.acc_no
WHERE c.grade = 'VIP';
EOF

+-----------+---------+----------+------------+
| cust_name | acc_no  | tx_type  | amount     |
+-----------+---------+----------+------------+
| 박지성    | 110-001 | DEPOSIT  | 1000000.00 |
| 박지성    | 110-001 | WITHDRAW |  500000.00 |
| 박지성    | 110-001 | DEPOSIT  | 1000000.00 |
| 박지성    | 110-001 | WITHDRAW |  500000.00 |
+-----------+---------+----------+------------+


### 실습 4. 계좌를 개설하지 않은 유휴 고객 식별 (IS NULL 활용)

**목적:** 마케팅 대상 선정을 위해 고객 테이블에는 존재하지만 계좌 테이블에는 데이터가 없는 고객을 추출합니다.

In [8]:
%%bash

mysql -u root --table bankdb << 'EOF'
SELECT c.cust_name, c.grade
FROM customers c
LEFT JOIN accounts a ON c.cust_id = a.cust_id
WHERE a.acc_no IS NULL;
EOF

+-----------+--------+
| cust_name | grade  |
+-----------+--------+
| 이강인    | SILVER |
+-----------+--------+


### 실습 5. 계좌별 총 거래 횟수 및 고객 정보 요약 (JOIN + GROUP BY)

**목적:** 각 계좌번호와 해당 계좌 소유주의 이름, 그리고 해당 계좌에서 발생한 총 거래 건수를 집계합니다.

In [9]:
%%bash

mysql -u root --table bankdb << 'EOF'
SELECT c.cust_name, a.acc_no, COUNT(t.tx_id) AS total_transactions
FROM customers c
JOIN accounts a ON c.cust_id = a.cust_id
LEFT JOIN transactions t ON a.acc_no = t.acc_no
GROUP BY c.cust_name, a.acc_no;
EOF

+-----------+---------+--------------------+
| cust_name | acc_no  | total_transactions |
+-----------+---------+--------------------+
| 김연아    | 110-004 |                  0 |
| 박지성    | 110-001 |                  4 |
| 박지성    | 110-002 |                  0 |
| 손흥민    | 110-003 |                  2 |
+-----------+---------+--------------------+


### 실습 6. 특정 일자 이후의 거래 발생 고객 조회 (INNER JOIN + Date Filter)

**[목적]** 2026년 3월 2일 이후에 거래를 수행한 고객의 이름과 거래 일자를 확인합니다.

- **Step 1 (연결):** 고객과 거래 내역을 계좌 번호를 징검다리 삼아 연결합니다.
- **Step 2 (필터):** 특정 날짜(`tx_date`) 조건을 적용합니다.

In [10]:
%%bash

mysql -u root --table bankdb << 'EOF'
SELECT c.cust_name, t.tx_date, t.amount
FROM customers c
INNER JOIN accounts a ON c.cust_id = a.cust_id
INNER JOIN transactions t ON a.acc_no = t.acc_no
WHERE t.tx_date >= '2026-03-02 00:00:00';
EOF

+-----------+---------------------+-----------+
| cust_name | tx_date             | amount    |
+-----------+---------------------+-----------+
| 박지성    | 2026-03-02 14:20:00 | 500000.00 |
| 손흥민    | 2026-03-03 09:15:00 | 200000.00 |
| 박지성    | 2026-03-02 14:20:00 | 500000.00 |
| 손흥민    | 2026-03-03 09:15:00 | 200000.00 |
+-----------+---------------------+-----------+


### 실습 7. 등급별 보유 계좌 총 잔액 합산 (LEFT JOIN + Group By)

**[목적]** 고객 등급별로 은행에 예치된 총 금액을 산출합니다. 계좌가 없는 등급도 0원으로 표시되어야 합니다.

- **Step 1 (연결):** 고객과 계좌를 연결하여 등급별 잔액 정보를 모읍니다.
- **Step 2 (집계):** `grade`를 기준으로 그룹화하고 `SUM` 연산을 수행합니다.

In [11]:
%%bash

mysql -u root --table bankdb << 'EOF'
SELECT c.grade, SUM(IFNULL(a.balance, 0)) AS total_grade_balance
FROM customers c
LEFT JOIN accounts a ON c.cust_id = a.cust_id
GROUP BY c.grade;
EOF

+--------+---------------------+
| grade  | total_grade_balance |
+--------+---------------------+
| GOLD   |          3000000.00 |
| SILVER |            50000.00 |
| VIP    |          5120000.00 |
+--------+---------------------+


### 실습 8. 입금(DEPOSIT) 거래만 발생한 계좌 정보 (JOIN + Constant Filter)

**[목적]** 출금 없이 입금 내역만 존재하는 계좌의 소유주와 금액을 확인합니다.

- **Step 1 (연결):** 고객-계좌-거래를 순차적으로 잇습니다.
- **Step 2 (필터):** 거래 유형(`tx_type`)이 'DEPOSIT'인 데이터만 남깁니다.

In [12]:
%%bash

mysql -u root --table bankdb << 'EOF'
SELECT c.cust_name, a.acc_no, t.amount
FROM customers c
JOIN accounts a ON c.cust_id = a.cust_id
JOIN transactions t ON a.acc_no = t.acc_no
WHERE t.tx_type = 'DEPOSIT';
EOF

+-----------+---------+------------+
| cust_name | acc_no  | amount     |
+-----------+---------+------------+
| 박지성    | 110-001 | 1000000.00 |
| 손흥민    | 110-003 |  200000.00 |
| 박지성    | 110-001 | 1000000.00 |
| 손흥민    | 110-003 |  200000.00 |
+-----------+---------+------------+


### 실습 9. 다수 계좌 보유 고객 식별 (Self-Join 개념 활용 대안)

**[목적]** 한 명의 고객이 두 개 이상의 계좌를 가진 경우를 찾아 고객명과 계좌 개수를 출력합니다.

- **Step 1 (연결):** 고객과 계좌를 연결합니다.
- **Step 2 (조건):** `HAVING` 절을 사용하여 계좌 수가 2개 이상인 그룹만 추출합니다.

In [13]:
%%bash

mysql -u root --table bankdb << 'EOF'
SELECT c.cust_name, COUNT(a.acc_no) AS acc_count
FROM customers c
JOIN accounts a ON c.cust_id = a.cust_id
GROUP BY c.cust_id, c.cust_name
HAVING COUNT(a.acc_no) >= 2;
EOF

+-----------+-----------+
| cust_name | acc_count |
+-----------+-----------+
| 박지성    |         2 |
+-----------+-----------+


### 실습 10. VIP 고객의 미사용 계좌 조회 (Complex Anti-Join)

**[목적]** 등급은 VIP인데, 거래 내역(`transactions`)이 전혀 없는 계좌를 찾아 관리 대상으로 분류합니다.

- **Step 1 (연결):** 고객-계좌를 `INNER JOIN`으로 묶어 VIP의 계좌만 먼저 추립니다.
- **Step 2 (확장):** 거래 내역을 `LEFT JOIN`으로 붙입니다.
- **Step 3 (추출):** 거래 내역이 `NULL`인 데이터만 골라냅니다.

In [14]:
%%bash

mysql -u root --table bankdb << 'EOF'
SELECT c.cust_name, a.acc_no
FROM customers c
JOIN accounts a ON c.cust_id = a.cust_id
LEFT JOIN transactions t ON a.acc_no = t.acc_no
WHERE c.grade = 'VIP' AND t.tx_id IS NULL;
EOF

+-----------+---------+
| cust_name | acc_no  |
+-----------+---------+
| 박지성    | 110-002 |
+-----------+---------+
